# 3.1 MDP의 다섯 요소와 마르코프 성질 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter03_1_mdp_elements.ipynb)

책 본문: [3.1 MDP의 다섯 요소와 마르코프 성질](https://smhanlab.com/book-ml/kor/ml2/chapter03/1.html)

이 노트북은 3.1절의 이론을 Gymnasium 실제 환경과 직접 만든 장난감 MDP로
숫자 확인합니다:

1. **3칸 보드게임 MDP**를 다섯 요소 $(S, A, P, R, \gamma)$로 통째로 정의
   하고, 본문 연습문제 5의 벨만방정식 해 $V(0) \approx 1.219$를
   반복 계산으로 확인.
2. **Gymnasium API로 $S, A$ 읽기** — FrozenLake(이산) vs CartPole(연속).
3. **마르코프 성질을 샘플링으로 검증** — 같은 상태 1에 두 경로로 도달한
   뒤의 다음 상태 분포가 같은지 확인.
4. **FrozenLake에서 $P$를 추정** — 무작위 탐색으로 경험적 전이확률 표를
   만들고, 알려진 물리(gymnasium 1.x 기본 성공률 1/3)와 비교.

## 0. 환경 준비

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print(f"numpy {np.__version__} | matplotlib {matplotlib.__version__} | 그림 저장: {IMG}")

numpy 2.4.6 | matplotlib 3.11.1 | 그림 저장: /home/smhan/book-ml/kor/src/images


## 1. 한 편의 MDP를 통째로 쓰기: 3칸 보드게임

본문과 같은 3-state MDP를 다섯 요소로 정의한다:

- $S = \{0, 1, 2\}$, 2는 터미널(self-loop)
- $A = \{$right, wait$\}$
- $P$: 0에서 right → 1(0.7)/0(0.3 미끄러짐), wait → 0; 1에서 right → 2,
  wait → 1; 2는 어디를 가도 자기 자신.
- $R$: 1에서 right(터미널 진입) = +3, 그 외 모든 스텝 = −1.
- $\gamma = 0.9$.

이 5-tuple 하나로 게임의 규칙 전체가 닫힌다. "항상 right" 정책의
기대 리턴 $V(0)$을 반복 계산으로 구하고, 본문 연습문제 5의 손계산
$V(0) \approx 1.219$와 비교한다.

In [2]:
gamma = 0.9

# P[s][a] = [(prob, next_state), ...]   R[s][a] = 즉시 보상
P = [
    [[(0.7, 1), (0.3, 0)], [(1.0, 0)]],    # state 0: [right, wait]
    [[(1.0, 2)], [(1.0, 1)]],              # state 1: [right, wait]
    [[(1.0, 2)], [(1.0, 2)]],              # state 2 (terminal, self-loop)
]
R = [
    [-1, -1],
    [+3, -1],   # 1에서 right(=터미널 진입)만 +3
    [0, 0],     # 터미널
]

def value_iteration(P, R, gamma, theta=1e-8, max_it=100000):
    """value iteration = 정책평가의 반복 고정점. 여기서는 정책이
    '항상 right'(a=0)로 고정되어 있어 정책평가와 같다."""
    n = len(P)
    V = [0.0] * n
    for it in range(max_it):
        delta = 0.0
        for s in range(n):
            a = 0
            v_new = R[s][a] + gamma * sum(p * V[sp] for p, sp in P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            break
    return V, it

V, iters = value_iteration(P, R, gamma)
for s in range(3):
    print(f"V({s}) = {V[s]:.4f}")
print(f"수렴: {iters}회 반복")

# 본문 연습문제 5 손계산: V(1)=3, V(0)=0.89/0.73≈1.219
hand = {2: 0.0, 1: 3.0, 0: 0.89 / 0.73}
assert abs(V[1] - 3.0) < 1e-4, f"V(1)={V[1]}"
assert abs(V[0] - hand[0]) < 1e-3, f"V(0)={V[0]} vs {hand[0]:.4f}"
print(f"✓ 손계산과 일치: V(0)≈{hand[0]:.3f} (미끄러짐 없는 버전은 1.7)")

V(0) = 1.2192
V(1) = 3.0000
V(2) = 0.0000
수렴: 16회 반복
✓ 손계산과 일치: V(0)≈1.219 (미끄러짐 없는 버전은 1.7)


### 1b. 전이 다이어그램 그림 생성 (본문에 삽입)

위 MDP를 matplotlib로 그린 SVG를 `kor/src/images/ch03_mdp_transition_diagram.svg`
로 저장한다 — 본문의 그림과 대응한다.

In [3]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9.0, 4.4))
ax.set_xlim(-1.4, 9.6)
ax.set_ylim(-1.9, 2.5)
ax.axis("off")

xs = {0: 0.0, 1: 4.0, 2: 8.0}
for s in (0, 1, 2):
    color = "#cdeacd" if s == 2 else "#dbeafe"
    ax.add_patch(mpatches.Circle((xs[s], 0), 0.62, fc=color, ec="#1f2937", lw=1.8, zorder=2))
    ax.text(xs[s], 0.09, f"s={s}", ha="center", va="center", fontsize=13, weight="bold", zorder=3)
    ax.text(xs[s], -0.26, "terminal" if s == 2 else "reward -1/step", ha="center", va="center",
            fontsize=8.5, color="#374151", zorder=3)

arrow_kw = dict(arrowstyle="-|>", lw=1.6, mutation_scale=16, shrinkA=0, shrinkB=0)

def arc(p1, p2, rad, color, ls, label, pos, fs=10):
    ax.add_patch(mpatches.FancyArrowPatch(
        p1, p2,
        connectionstyle=f"arc3,rad={rad}",
        color=color, linestyle=ls, zorder=1, **arrow_kw))
    ax.text(pos[0], pos[1], label, ha="center", va="center", fontsize=fs, color=color)

# 0 -> 1 : right (실선, 위로 오목한 호)
arc((xs[0] + 0.6, 0.18), (xs[1] - 0.6, 0.18), -0.35, "#1d4ed8", "-",
    "right: P=0.7", (2.0, 1.15))
# 1 -> 0 : 미끄러짐 (실선, 아래로 오목한 호)
arc((xs[1] - 0.6, -0.18), (xs[0] + 0.6, -0.18), -0.35, "#1d4ed8", "-",
    "slip: P=0.3", (2.0, -1.05))
# 0 -> 0 : wait (자기 자신, 위쪽 반원)
arc((xs[0], 0.6), (xs[0], 0.6), 1.6, "#6b7280", "--",
    "wait", (xs[0], 1.75), fs=9)
# 1 -> 2 : right +3 (실선)
arc((xs[1] + 0.6, 0.18), (xs[2] - 0.6, 0.18), -0.35, "#1d4ed8", "-",
    "right: P=1, reward +3", (6.0, 1.15))
# 1 -> 1 : wait (자기 자신, 위쪽 반원)
arc((xs[1], 0.6), (xs[1], 0.6), 1.6, "#6b7280", "--",
    "wait", (xs[1], 1.75), fs=9)
# 2 -> 2 : self-loop (위쪽 반원)
arc((xs[2], 0.6), (xs[2], 0.6), 1.6, "#047857", "--",
    "self-loop (reward 0)", (xs[2], 1.75), fs=9)

ax.text(4.0, -1.6, "solid = right (stochastic), dashed = wait / terminal self-loop,  γ = 0.9",
        ha="center", fontsize=9.5, color="#374151")
ax.set_title("3-state board-game MDP — the entire ruleset is captured by (S, A, P, R, γ)", fontsize=12, pad=12)

svg_path = os.path.join(IMG, "ch03_mdp_transition_diagram.svg")
fig.savefig(svg_path, bbox_inches="tight")
plt.close(fig)
print("저장:", svg_path)

저장: /home/smhan/book-ml/kor/src/images/ch03_mdp_transition_diagram.svg


## 2. Gymnasium에서 $S, A$를 실제로 읽기

이론의 다섯 요소를 실제 환경의 API와 대응시킨다. FrozenLake는
$|S|=16, |A|=4$의 **이산** MDP( $P$를 표로 쓸 수 있음),
CartPole은 $S \subset \mathbb{R}^4$ **연속 상태** + 2개 이산 행동
($P$를 표로 쓸 수 없음) — 표 기반(Chapter 4~6)과 신경망 기반
(Chapter 9~11)의 분리가 **환경 수준에서** 보이는 것이다.

In [4]:
import gymnasium as gym

# --- FrozenLake: 이산 상태·행동 공간 ---
fl = gym.make("FrozenLake-v1", is_slippery=True)
obs, info = fl.reset(seed=0)
print("FrozenLake 상태 공간 S:", fl.observation_space)   # Discrete(16)
print("FrozenLake 행동 공간 A:", fl.action_space)         # Discrete(4)
print("초기 상태:", obs, "(4x4 격자 16칸 중 하나)")
fl.close()

print()

# --- CartPole: 연속 상태, 이산 행동 ---
cp = gym.make("CartPole-v1")
obs, info = cp.reset(seed=0)
print("CartPole 상태 공간 S:", cp.observation_space)    # Box(4,)
print("CartPole 행동 공간 A:", cp.action_space)          # Discrete(2)
print("초기 상태:", obs, "[x, x_dot, theta, theta_dot]")
cp.close()

FrozenLake 상태 공간 S: Discrete(16)
FrozenLake 행동 공간 A: Discrete(4)
초기 상태: 0 (4x4 격자 16칸 중 하나)

CartPole 상태 공간 S: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
CartPole 행동 공간 A: Discrete(2)
초기 상태: [ 0.01369617 -0.02302133 -0.04590265 -0.04834723] [x, x_dot, theta, theta_dot]


## 3. 마르코프 성질을 샘플링으로 검증

마르코프 성질 $P(s_{t+1} \mid s_t, a_t, s_{t-1}, a_{t-1}, \ldots) =
P(s_{t+1} \mid s_t, a_t)$가 "성립한다"는 말이 검증 가능한 주장인지
직접 본다. 2-경로 결정적 환경(상태 0에서 행동 0이면 1로, 행동 1이면 2로
전이; 1은 self-loop)에서, 상태 1에 **다른 경로**로 도달한 뒤의
다음 상태 분포가 같은지 $N$회 샘플링으로 비교한다.

In [5]:
class TwoPathsEnv:
    """상태 0 -> {1, 2} -> 1 -> 1(self-loop) 구조의 결정적 MDP."""
    def __init__(self, seed=0):
        self.rng = np.random.default_rng(seed)
        self.s = 0

    def reset(self, seed=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.s = 0
        return self.s

    def step(self, a):
        # 상태 0: a=0 -> 1, a=1 -> 2
        # 상태 1: 어디서 오든 self-loop (마르코프: 경로 무관)
        # 상태 2: a=0 -> 1, a=1 -> 2(self)
        if self.s == 0:
            self.s = 1 if a == 0 else 2
        elif self.s == 1:
            self.s = 1  # self-loop, 경로 무관
        else:
            self.s = 1 if a == 0 else 2
        return self.s, 0.0, False, False, {}

# 마르코프 검증: 상태 1에 두 경로로 도달, 다음 전이 분포 비교
rng = np.random.default_rng(42)
dist_via_01 = []   # 0 -> 1 경로를 거쳐 1에 도달
dist_via_021 = []  # 0 -> 2 -> 1 경로를 거쳐 1에 도달
N = 10000

for _ in range(N):
    env = TwoPathsEnv(seed=rng.integers(0, 2**31))
    env.reset()
    env.step(0)  # 0 -> 1
    env.step(0)  # 1 -> 1, 다음 전이 샘플
    dist_via_01.append(env.s)

for _ in range(N):
    env = TwoPathsEnv(seed=rng.integers(0, 2**31))
    env.reset()
    env.step(1)  # 0 -> 2
    env.step(0)  # 2 -> 1
    env.step(0)  # 1 -> 1, 다음 전이 샘플
    dist_via_021.append(env.s)

p1 = np.mean(np.array(dist_via_01) == 1)
p2 = np.mean(np.array(dist_via_021) == 1)
print(f"경로 0->1    : P(다음=1 | s=1) ≈ {p1:.4f}")
print(f"경로 0->2->1 : P(다음=1 | s=1) ≈ {p2:.4f}")
print(f"차이: {abs(p1-p2):.4f}  (마르코프 성질: 0이어야 함)")
assert abs(p1 - p2) < 0.01, "마르코프 성질 위반!"
print("✓ 두 경로에서 다음 상태 분포가 일치 — 마르코프 성질 확인")

경로 0->1    : P(다음=1 | s=1) ≈ 1.0000
경로 0->2->1 : P(다음=1 | s=1) ≈ 1.0000
차이: 0.0000  (마르코프 성질: 0이어야 함)
✓ 두 경로에서 다음 상태 분포가 일치 — 마르코프 성질 확인


## 4. FrozenLake에서 $P$를 샘플링으로 추정

Gymnasium 환경의 $P$는 `step()` 내부에 감춰져 있어 **샘플**만 볼 수
있다. $P(s'|s,a) \approx N(s,a,s')/N(s,a)$로, 무작위 행동을 많이 하면
경험적 전이확률에 수렴한다. 2000에피소드의 무작위 탐색 후,
대표 $(s,a)$ 셀의 추정값을 실제 물리와 비교한다. (참고: 본문이 인용한
구버전 물리는 intended 2/3, 좌/우 1/6씩이지만, 이 노트북이 실행되는
gymnasium 1.x의 기본 성공률은 1/3 — intended 1/3, 수직 방향 1/3씩.)
이 $P_{\text{est}}$를 Chapter 4의 동적계획법에 대입하면 **모델 기반**,
$P$를 무시하고 경험만 쓰면 **모델 없는** 알고리즘이 된다 —
같은 환경·샘플을 쓰는 두 가족의 갈림길.

In [6]:
import gymnasium as gym

env = gym.make("FrozenLake-v1", is_slippery=True)
n_states = env.observation_space.n   # 16
n_actions = env.action_space.n       # 4
# count[s][a][s'] = (s,a)->s' 전이 횟수
count = np.zeros((n_states, n_actions, n_states), dtype=int)

for ep in range(2000):
    s, _ = env.reset(seed=ep)
    done = False
    while not done:
        a = env.action_space.sample()
        s2, r, term, trunc, _ = env.step(a)
        if s != 15 and s2 != 15:  # 터미널 상태 전이 제외
            count[s, a, s2] += 1
        s = s2
        done = term or trunc
env.close()

tot = count.sum(axis=2, keepdims=True)
P_est = np.where(tot > 0, count / np.maximum(tot, 1), 0.0)
print("P_est shape:", P_est.shape, " (16 상태 × 4 행동 × 16 다음 상태)")
print(f"관측된 (s,a) 셀: {(count.sum(axis=2) > 0).sum()}/{n_states*n_actions}, 총 전이: {count.sum()}")
print("  (나머지 셀은 벽 밖 전이/도달 곤란 칸 — 무작위 탐색의 자연스러운 공백)")

# 대표 셀: 상태 0(시작 칸)에서 left. 벽 밖으로 가는 이동은 "자리참"으로
# 합쳐지기 때문에 2개 상태(0, 4)에만 질량이 몰린다 — 경계 효과.
nz = np.nonzero(P_est[0, 0])[0]
print(f"P_est[0, 0] (상태 0에서 left) nonzero: {nz.tolist()}")
print(f"  경험 확률: {np.round(P_est[0, 0][nz], 3).tolist()}")
print("  (gymnasium 1.x 성공률 1/3 물리: intended·slip이 벽 밖이면 전부 '0에 잔다')")

P_est shape: (16, 4, 16)  (16 상태 × 4 행동 × 16 다음 상태)
관측된 (s,a) 셀: 44/64, 총 전이: 14827
  (나머지 셀은 벽 밖 전이/도달 곤란 칸 — 무작위 탐색의 자연스러운 공백)
P_est[0, 0] (상태 0에서 left) nonzero: [0, 4]
  경험 확률: [0.66, 0.34]
  (gymnasium 1.x 성공률 1/3 물리: intended·slip이 벽 밖이면 전부 '0에 잔다')


## 정리

**강화학습 문제 = $(S, A, P, R, \gamma)$ + 마르코프 성질.**

1. 3칸 보드게임: 5-tuple 하나로 규칙 전체가 닫히고, "항상 right"의
   $V(0) \approx 1.219$는 벨만방정식 손계산(연습문제 5)과 일치.
2. $S, A$는 환경의 `observation_space`/`action_space`로 읽힌다 —
   이산이면 표 기반, 연속이면 신경망 기반 알고리즘으로 갈라진다.
3. 마르코프 성질은 샘플링으로 **검증 가능한 주장**이다 — 같은 상태에
   다른 경로로 도달해도 다음 상태 분포가 같아야 한다.
4. $P$는 `step()` 안에서 감겨 있지만, 충분한 샘플로 경험적 표를
   추정할 수 있다 — 그것이 모델 기반/모델 없는 분리의 실질적 의미.